In [13]:
import h5py
import numpy as np
f = h5py.File('C:/Users/zuohaonan/Desktop/PE4/NN_accelerator/minst_CNN_weights04.h5','r')
#t = h5py.File('C:/Users/zuohaonan/Desktop/PE4/NN_accelerator/data_test04.h5','r')
t = h5py.File('C:/Users/zuohaonan/Desktop/PE4/NN_accelerator/data_test04.h5','r')

kernel = [data.decode('utf-8') for data in f['kernel_weights']]
bias = [data.decode('utf-8') for data in f['bias']]
# pool_weight = [data.decode('utf-8') for data in f['pool_weights']]
# pool_bias = [data.decode('utf-8') for data in f['pool_bias']]
    
test_data = [data.decode('utf-8') for data in t['in_data']]

print("卷积核权重：\n%s\n" % kernel)
print("六个偏置：\n%s\n" % bias)
# print("六个池化权重：\n%s\n" % pool_weight)
# print("六个池化偏置：\n%s\n" % pool_bias)
print("输入数据：\n%s\n" % test_data)

卷积核权重：
['1111111111111111', '0000000000000010', '0000000000000111', '0000000000000011', '0000000000000000', '1111111111111111', '0000000000000000', '1111111111111010', '0000000000000010', '0000000000000100', '1111111111111000', '1111111111111100', '1111111111111101', '0000000000000110', '1111111111111101', '1111111111111110', '1111111111111010', '1111111111111110', '0000000000000010', '0000000000000001', '0000000000000110', '0000000000000010', '0000000000000111', '1111111111111111', '0000000000000001', '0000000000000010', '1111111111111011', '1111111111111111', '1111111111111000', '1111111111111010', '1111111111111000', '1111111111111110', '1111111111111110', '1111111111111010', '1111111111111101', '0000000000000001', '0000000000000010', '1111111111111100', '0000000000000110', '1111111111111101', '0000000000000110', '0000000000000011', '0000000000000100', '1111111111111110', '1111111111111010', '0000000000000011', '0000000000000101', '1111111111111010', '1111111111111111', '11111111111

In [21]:
import h5py
import numpy as np
##f = h5py.File('/Users/zuohaonan/Desktop/CNN/python/minst_CNN_weights2.h5','r')
##t = h5py.File('/Users/zuohaonan/Desktop/CNN/python/data_test2.h5','r')

kernel = [data.decode('utf-8') for data in f['kernel_weights']]
    
test_data = [data.decode('utf-8') for data in t['in_data']]

source = open('C:\\Users\\zuohaonan\\Desktop\\PE4\\NN_accelerator\\source_CNN04.txt','w')
##print('memory_initialization_radix=16;\nmemory_initialization_vector=', file=source)
pe_id = '0000000000000001'

for i in range(0,32):
    for j in range(0,64):##同一个核的权重排列在一起
        s = '00' + pe_id + '1000000' + '{:0>7b}'.format(j+1) + kernel[i+32*j]
        print('{:0>12}'.format(hex(int(s, 2))[2:]), file=source)

for i in range(0,64):
    s = '00' + pe_id + '1000000' + '1000001' + bias[i][24:40]
    print('{:0>12}'.format(hex(int(s, 2))[2:]), file=source)
    s = '00' + pe_id + '1000000' + '1000010' + bias[i][8:24]
    print('{:0>12}'.format(hex(int(s, 2))[2:]), file=source)
    s = '00' + pe_id + '1000000' + '1000011' +  '00000000' + bias[i][0:8]
    print('{:0>12}'.format(hex(int(s, 2))[2:]), file=source)

for i in range(0,32):
    s = '00' + pe_id + '000' + '{:0>11b}'.format(i) + (test_data[i])
    print(s)
    print('{:0>12}'.format(hex(int(s, 2))[2:]), file=source)
    if i == 31:
        print(s)
        print('{:0>12}'.format(hex(int(s, 2))[2:]), file=source)

000000000000000001000000000000000000000000000100
000000000000000001000000000000011111111111111110
000000000000000001000000000000100000000000000111
000000000000000001000000000000111111111111111000
000000000000000001000000000001000000000000000000
000000000000000001000000000001010000000000000111
000000000000000001000000000001101111111111111111
000000000000000001000000000001110000000000000011
000000000000000001000000000010001111111111111101
000000000000000001000000000010010000000000000100
000000000000000001000000000010100000000000000111
000000000000000001000000000010111111111111111111
000000000000000001000000000011001111111111111010
000000000000000001000000000011010000000000000000
000000000000000001000000000011100000000000000000
000000000000000001000000000011110000000000000001
000000000000000001000000000100000000000000000010
000000000000000001000000000100010000000000000100
000000000000000001000000000100100000000000000010
000000000000000001000000000100111111111111111100
00000000000000000100

In [22]:
import numpy as np
import pandas as pd
#中位数为26
def trans(str):
    num = int(str, 2)

    if num > 32767:
        # 负数，计算补码
        value =num-65536
    else:
        value = num
    return value

def trans40(str):
    num = int(str, 2)

    if num > 549755813887:
        # 负数，计算补码
        value =num-1099511627776
    else:
        value = num
    return value

def binary_to_decimal(number):

    if number > 32767:
        # 负数，计算补码
        value =number-65536
    else:
        value = number
    return value

def cut_sobel(dacc, bit):
    mask = (1 << (bit-15)) - 1  # 0b11111111111111111111111111

    temp_bits = (dacc >> 15) & mask  # 获取27-40位
    dacc_40 = (dacc >> (bit-1)) & 1  # 获取最高位
    dacc_li = dacc & ((1 << 15) - 1)  # 获取0-14位

    if temp_bits == 0 or temp_bits == (1 << (bit-15)) - 1:
        dcut = (dacc_40 << 15) | dacc_li
    else:
        dcut = 0b1000000000000000 if dacc_40 else 0b0111111111111111
    return (binary_to_decimal(dcut))

test_data_convert = list(map(trans, test_data))
IFM = np.array(test_data_convert)
#IFM = test_data_np.reshape(5,5,32)##IFM是原输入图，按优先级reshape

#mapp = np.zeros((32,5,5), dtype=np.int16)#8通道，14行14列
#mapp = np.transpose(IFM, (2, 0, 1))#调整轴的顺序为通道、行、列
            
print('原输入特征图：')
print(IFM)

count = 0
for k in range(32):
    if IFM[k] == 0:
        count = count + 1       

print('稀疏度：')
print(count)

kernel_convert = list(map(trans, kernel))
ker = np.zeros((64,32), dtype=np.int16)

Core = [np.zeros((32,3,3), dtype=int) for _ in range(32)]
kernel_re = [np.zeros((3,3,32), dtype=int) for _ in range(32)]

for i in range(0,64):
    for j in range(0,32):
        ker[i][j] = kernel_convert[j + 32 * i]
        
for k in range(0,64):
    print(f"这是第{k}个卷积核:")
    print(ker[k])

bias_convert = list(map(trans40, bias))
print('偏置:')
print(bias_convert)

#post_MAC = [np.zeros((3, 3), dtype=int) for _ in range(32)]
#mem = [np.zeros((3, 3), dtype=int) for _ in range(32)]
mem = np.zeros(64, dtype=int)
OFM = np.zeros(64, dtype=int)

def conv(num):
    result = 0
    for i in range(0,32):
        result += (ker[num])[i]*IFM[i]

    mem[num] =result
    result += int(bias_convert[num])

    if result < 0:
        result = 0
            
    return result

def cut(dacc,bit):

    mask= (1 << (26-bit)) - 1  # 0b11111111111111111111111111
    
    temp_bits = (dacc>> (bit+15)) & mask#获取27-40位
    dacc_40 = (dacc >> 40) & 1#获取最高位
    dacc_li = (dacc>>bit) & ((1 << 15) - 1)#获取0-14位

    if temp_bits == 0 or temp_bits == (1 << (26-bit)) - 1:
        dcut = (dacc_40 << 15) | dacc_li
    else:
        dcut = 0b1000000000000000 if dacc_40 else 0b0111111111111111
    return (binary_to_decimal(dcut))

for i in range(0,64):
    OFM[i] = conv(i)

all_values = np.concatenate([arr.flatten() for arr in OFM])
non_zero_values = all_values[all_values != 0]  # 过滤非零值

# 计算统计值（包括全体的最大值和最小值）
max_value = np.max(all_values)  # 包含零的最大值
min_value = np.min(all_values)  # 包含零的最小值

# 计算非零值的平均值和中位数
if len(non_zero_values) > 0:
    mean_non_zero = np.mean(non_zero_values)
    median_non_zero = np.median(non_zero_values)
else:
    mean_non_zero = 0  # 如果无非零值，默认输出0
    median_non_zero = 0

# 输出结果
print(f"最大值（含零）: {max_value}")
print(f"最小值（含零）: {min_value}")
print(f"非零值的平均值: {mean_non_zero}")
print(f"非零值的中位数: {median_non_zero}")

print(f'这是中间结果')
print(mem)
print('\n')


#for k in range(0,32):
#    OFM[k]= pool(0,0,k)

with open('C:\\Users\\zuohaonan\\Desktop\\PE4\\NN_accelerator\\ofm_data04.txt', 'w') as file:
    print(f'这是输出向量')
    print(OFM)
    print('\n')
    for k in range(0,64):
        if OFM[k] < 0:
            value = OFM[k] + 65536
        else:
            value = OFM[k]
            
        hex_value = format(value, '04X')
        file.write(hex_value + '\n')

with h5py.File('C:\\Users\\zuohaonan\\Desktop\\PE4\\NN_accelerator\\resultl4.h5', 'w') as hf:
    # 初始化result1数据集（1568个UTF-8编码的16位字符串）
    result1 = hf.create_dataset("result", 
                               shape=(64,), 
                               dtype=h5py.string_dtype(encoding='utf-8', length=16))  # UTF-8固定长度16字节
    
    index = 0  # 写入位置索引
    
    # 遍历OFM并填充result1
    

    for k in range(64):
        # 处理负数（转换为补码形式的正数）
        value = OFM[k]
        if value > 18:
            value = int(value/18)  # 16位补码转换
                
        # 转换为16位二进制字符串（UTF-8格式）
        binary_str = format(value, '016b')  # 如 "0000000000000000"
        result1[index] = binary_str
        index += 1

print("已写入完毕")

原输入特征图：
[ 4 -2  7 -8  0  7 -1  3 -3  4  7 -1 -6  0  0  1  2  4  2 -4 -5 -7 -6  7
 -2 -2 -8  3  2  4 -8 -3]
稀疏度：
3
这是第0个卷积核:
[-1  2  7  3  0 -1  0 -6  2  4 -8 -4 -3  6 -3 -2 -6 -2  2  1  6  2  7 -1
  1  2 -5 -1 -8 -6 -8 -2]
这是第1个卷积核:
[-2 -6 -3  1  2 -4  6 -3  6  3  4 -2 -6  3  5 -6 -1 -8  1  1 -7 -2 -5 -6
  4  6  7 -1  6  0  4  2]
这是第2个卷积核:
[ 7 -8 -5  0  1  5  6 -1 -7  6 -4  1 -7  5  2 -6  5  2  7  0  0 -7  0  6
 -6  3 -2 -6 -3  6  6  3]
这是第3个卷积核:
[ 2  6  2 -7 -3  5  7 -3  5  1 -4  0 -7  6  3 -7 -2  5 -2  5  3 -1 -5  4
  3  7 -7 -5 -3 -5 -2  3]
这是第4个卷积核:
[ 6  6  7  5 -8  5  4  2  6 -7  2 -6  7 -8  6  2 -7 -2 -8  5  5 -2  6  4
  3  5 -8  1  7 -1  7  0]
这是第5个卷积核:
[ 6 -3 -8  6  1  1  0 -6 -8 -7  1 -3  0 -4  4  7 -8  6  0  6  7 -3  6 -1
  5 -5  5  4 -3  0  6 -3]
这是第6个卷积核:
[-2  1 -2 -3  1 -7 -3  5  2 -3  1 -5  4 -5 -8  2  6 -6 -5  1  4  1  2 -1
  2 -1  4  1 -1  1 -3 -8]
这是第7个卷积核:
[-1  4 -8 -5 -2 -7  7  4  1  1 -7 -6  6  2 -6  1 -4 -4 -6 -3  4 -6  0 -6
  3 -7  4  7  0  4 -7 -2]
这是第8个卷积核:
[-3 